# Dataset Manifest Generator

This notebook generates a manifest for the full train dataset. Each row maps one image to its label, batch, and split. Labels support one or more percentages, for example `R5` or `R5 S10` (rice hull 5%, salt 10%).

Keep the same output keys throughout one training run. For rice hull and salt, use `R5 S0` when salt is absent. Output keys are sorted consistently, so `R5 S10` and `S10 R5` both map to `[5, 10]` in `R, S` order. Keep the original folder label unchanged for prepared-image paths.

## Manifest format

| image_path                      | label | batch   | split      |
|:--------------------------------|:-----:|:--------|:----------:|
| dataset/R5/batch_1/img_001.jpg  | R5    | batch_1 | training   |
| dataset/R0/batch_2/img_001.jpg  | R0    | batch_2 | testing    |
| dataset/R10/batch_3/img_001.jpg | R10   | batch_3 | validation |

In [ ]:
from pathlib import Path
import pandas as pd
from utils.labels import infer_label_keys

In [2]:
DATASET_DIR = Path(r"C:\Users\realb\Documents\PROJECT_FILES\Dataset\90mm")

Records the ```image_path```, ```label```, and ```batch```

In [ ]:
records = []

for image in DATASET_DIR.rglob("*"):
    if not image.is_file():
        continue
    if image.suffix.lower() not in {".jpg", ".jpeg", ".png"}:
        continue

    records.append({
        "image_path": str(image),
        "label": image.parent.parent.name,
        "batch": image.parent.name,
    })
    
if not records:
    raise ValueError("IMAGE WHEREE???????")

df = pd.DataFrame(records)
label_keys = infer_label_keys(df["label"])
print("Output order:", label_keys)
print(df.head())

Splits the batches in their ```split``` calculated 70% training 15% testing 15% validation, splitted by batch count, not image count

In [5]:
batches = df[["label", "batch"]].drop_duplicates().copy()
batches["split"] = "" # split are training, validation, and testing

for label in batches["label"].unique():
    mask = batches["label"] == label 
    group = batches[mask].sample(frac=1, random_state=42)
    
    batches_per_label = len(group)
    
    if batches_per_label < 3:
        raise ValueError(f"Not enough batches for label '{label}' to split into training, validation, and testing.")
    
    train_batches = min(round(batches_per_label * 0.7), batches_per_label - 2)
    remaining_batches = batches_per_label - train_batches
    validation_batches = remaining_batches // 2
    test_batches = remaining_batches - validation_batches
    
    print(f"Label: {label}, Total Batches: {batches_per_label}, Train: {train_batches}, Validation: {validation_batches}, Test: {test_batches}")
    
    batches.loc[group.index, "split"] = (
        ["training"] * train_batches
        + ["validation"] * validation_batches
        + ["testing"] * test_batches
    )
    
df = df.merge(batches, on=["label", "batch"], validate="many_to_one")

Label: R5, Total Batches: 10, Train: 7, Validation: 1, Test: 2
Label: R10, Total Batches: 10, Train: 7, Validation: 1, Test: 2
Label: R0, Total Batches: 10, Train: 7, Validation: 1, Test: 2


In [6]:
print(pd.crosstab(batches["label"], batches["split"]))
print(pd.crosstab(df["label"], df["split"]))
df.to_csv("manifest.csv", index=False)

split  testing  training  validation
label                               
R0           2         7           1
R10          2         7           1
R5           2         7           1
split  testing  training  validation
label                               
R0          40       140          20
R10         40       140          20
R5          40       140          20
